# IT3091 - Member 2 - IT24100533 - Danthanarayana D.M.R


## Member 2 - Weather and location data

In [21]:
!pip -q install openpyxl joblib

import os
import re
import json
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

COLAB_REPO = Path("/content/IT3091---Machine-Learning-Assignment")
REPO_PATH = COLAB_REPO if COLAB_REPO.exists() else Path.cwd()  # Colab, or local (notebook opened from repo folder)
RAW_PATH = REPO_PATH / "Raw Datasets"
PREPROCESSED_PATH = REPO_PATH / "Preprocessed Datasets"

PREPROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print("Raw data:", RAW_PATH)
print("Preprocessed data:", PREPROCESSED_PATH)

Raw data: /Users/ravindudanthanarayana/Desktop/IT3091---Machine-Learning-Assignment/Raw Datasets
Preprocessed data: /Users/ravindudanthanarayana/Desktop/IT3091---Machine-Learning-Assignment/Preprocessed Datasets


In [22]:
WEATHER_FILE = RAW_PATH / "weatherData.csv"
LOCATION_FILE = RAW_PATH / "locationData.csv"

if not WEATHER_FILE.exists():
    matches = list(RAW_PATH.glob("*weatherData*.csv"))
    if matches:
        WEATHER_FILE = matches[0]

if not LOCATION_FILE.exists():
    matches = list(RAW_PATH.glob("*locationData*.csv"))
    if matches:
        LOCATION_FILE = matches[0]

if not WEATHER_FILE.exists() or not LOCATION_FILE.exists():
    raise FileNotFoundError("weatherData.csv or locationData.csv was not found in Raw Datasets.")

print("Weather file:", WEATHER_FILE.name)
print("Location file:", LOCATION_FILE.name)

Weather file: weatherData.csv
Location file: locationData.csv


In [23]:
weather = pd.read_csv(WEATHER_FILE)
locations = pd.read_csv(LOCATION_FILE)

weather = weather.loc[:, ~weather.columns.astype(str).str.lower().isin(["index", "unnamed: 0"])]
locations = locations.loc[:, ~locations.columns.astype(str).str.lower().isin(["index", "unnamed: 0"])]

weather = weather.merge(
    locations[["location_id", "city_name"]],
    on="location_id",
    how="left",
    validate="many_to_one"
)

if weather["city_name"].isna().any():
    raise ValueError("Some location_id values did not match locationData.csv.")

weather["date"] = pd.to_datetime(weather["date"], errors="coerce")
weather = weather.dropna(subset=["date"]).copy()

print("Weather shape:", weather.shape)
display(weather.head())

Weather shape: (142371, 22)


,location_id,date,weather_code (wmo code),temperature_2m_max (°C),temperature_2m_min (°C),temperature_2m_mean (°C),apparent_temperature_max (°C),apparent_temperature_min (°C),apparent_temperature_mean (°C),daylight_duration (s),sunshine_duration (s),precipitation_sum (mm),rain_sum (mm),precipitation_hours (h),wind_speed_10m_max (km/h),wind_gusts_10m_max (km/h),wind_direction_10m_dominant (°),shortwave_radiation_sum (MJ/m²),et0_fao_evapotranspiration (mm),sunrise,sunset,city_name
0,0,2010-01-01,1,30.1,22.6,26.0,34.5,25.0,29.0,42220.20,38905.73,0.0,0.0,0,12.2,27.4,19,20.92,4.61,06:22,18:05,Colombo
1,0,2010-01-02,51,30.1,23.7,26.3,33.9,26.1,29.7,42225.71,37451.01,0.1,0.1,1,13.0,27.0,24,17.71,3.91,06:22,18:06,Colombo
2,0,2010-01-03,51,29.6,23.1,26.0,34.5,26.2,29.9,42231.68,33176.43,0.6,0.6,3,12.3,27.4,17,17.76,3.66,06:22,18:06,Colombo
3,0,2010-01-04,2,28.9,23.1,25.7,31.7,26.1,28.4,42238.11,38289.20,0.0,0.0,0,17.0,34.6,357,16.50,3.75,06:23,18:07,Colombo
4,0,2010-01-05,1,28.1,21.3,24.6,30.0,22.9,26.2,42244.99,39113.82,0.0,0.0,0,18.7,37.1,353,23.61,5.00,06:23,18:07,Colombo


In [24]:
def season_and_year(dt):
    m, y = dt.month, dt.year

    if m in [4, 5, 6, 7, 8]:
        return pd.Series(["Yala", y])

    if m in [9, 10, 11, 12]:
        return pd.Series(["Maha", y])

    if m in [1, 2, 3]:
        return pd.Series(["Maha", y - 1])

    return pd.Series([pd.NA, pd.NA])

weather[["Season", "Year"]] = weather["date"].apply(season_and_year)
weather["Year"] = pd.to_numeric(weather["Year"], errors="coerce").astype("Int64")
weather = weather[weather["Year"].between(2012, 2023)].copy()

display(weather[["date", "Season", "Year"]].head())

,date,Season,Year
821,2012-04-01,Yala,2012
822,2012-04-02,Yala,2012
823,2012-04-03,Yala,2012
824,2012-04-04,Yala,2012
825,2012-04-05,Yala,2012


In [25]:
# Every numeric weather column is kept (sunrise/sunset are time-of-day text, so they are dropped)
SUM_COLS = [
    "daylight_duration (s)", "sunshine_duration (s)", "precipitation_sum (mm)", "rain_sum (mm)",
    "precipitation_hours (h)", "shortwave_radiation_sum (MJ/m²)", "et0_fao_evapotranspiration (mm)"
]
MEAN_COLS = [
    "temperature_2m_max (°C)", "temperature_2m_min (°C)", "temperature_2m_mean (°C)",
    "apparent_temperature_max (°C)", "apparent_temperature_min (°C)", "apparent_temperature_mean (°C)",
    "wind_speed_10m_max (km/h)", "wind_gusts_10m_max (km/h)", "wind_direction_10m_dominant (°)"
]
CODE_COL = "weather_code (wmo code)"
WEATHER_COLS = SUM_COLS + MEAN_COLS + [CODE_COL]

for c in WEATHER_COLS:
    weather[c] = pd.to_numeric(weather[c], errors="coerce")

def clean(col):
    name = re.sub(r"\s*\(.*?\)", "", col)
    unit = re.search(r"\((.*?)\)", col)
    unit = re.sub(r"[^0-9A-Za-z]+", "", unit.group(1).replace("²", "2")) if unit else ""
    name = "".join(w.capitalize() for w in re.split(r"[_\s]+", name.strip()))
    return name + (f"_{unit}" if unit else "")

def summarise(df, group_cols):
    agg = {}
    for c in SUM_COLS:
        agg["Total" + clean(c)] = (c, "sum")
    for c in MEAN_COLS:
        agg["Mean" + clean(c)] = (c, "mean")
    agg["MostCommonWeatherCode"] = (CODE_COL, lambda s: s.mode().iloc[0] if s.notna().any() else np.nan)
    agg["WeatherDays"] = ("date", "nunique")
    return df.groupby(group_cols, as_index=False).agg(**agg)

# District level (one row per city / Year / Season)
weather_district = summarise(weather.rename(columns={"city_name": "District"}), ["District", "Year", "Season"])

# National level (average across all cities per day first, then summarised per Year / Season)
weather_daily_nat = weather.groupby(["date", "Season", "Year"], as_index=False)[WEATHER_COLS].mean()
weather_national = summarise(weather_daily_nat, ["Year", "Season"])
weather_national.insert(0, "District", "National Total")

weather_season = (
    pd.concat([weather_national, weather_district], ignore_index=True)
      .sort_values(["District", "Year", "Season"]).reset_index(drop=True)
)

print("Shape:", weather_season.shape)
display(weather_season.head(10))

Shape: (672, 21)


,District,Year,Season,TotalDaylightDuration_s,TotalSunshineDuration_s,TotalPrecipitationSum_mm,TotalRainSum_mm,TotalPrecipitationHours_h,TotalShortwaveRadiationSum_MJm2,TotalEt0FaoEvapotranspiration_mm,MeanTemperature2mMax_C,MeanTemperature2mMin_C,MeanTemperature2mMean_C,MeanApparentTemperatureMax_C,MeanApparentTemperatureMin_C,MeanApparentTemperatureMean_C,MeanWindSpeed10mMax_kmh,MeanWindGusts10mMax_kmh,MeanWindDirection10mDominant,MostCommonWeatherCode,WeatherDays
0,Ampara,2012,Maha,9086782.62,6940951.31,1576.0,1576.0,1666.0,3645.61,778.46,29.116038,24.067925,26.137264,34.016038,28.047642,30.287264,14.373585,32.987736,136.443396,51.0,212
1,Ampara,2012,Yala,6843710.30,5945818.10,220.1,220.1,285.0,3335.31,799.91,33.631373,25.779085,29.147059,39.181046,29.817647,33.445098,13.405229,34.403268,197.000000,51.0,153
2,Ampara,2013,Maha,9086844.10,7462366.94,935.9,935.9,1353.0,3860.26,828.13,29.243868,23.830189,26.183962,33.582547,27.545283,29.960849,15.209906,34.916981,129.679245,51.0,212
3,Ampara,2013,Yala,6843639.31,5765720.58,271.5,271.5,312.0,3237.43,749.10,32.962092,25.666013,28.735294,38.658824,30.113725,33.364706,12.525490,34.275163,191.150327,2.0,153
4,Ampara,2014,Maha,9086907.27,7103198.45,1635.8,1635.8,1854.0,3758.94,776.61,28.887736,23.675000,25.868396,33.971698,27.715566,30.158491,13.413679,31.482547,133.547170,63.0,212
5,Ampara,2014,Yala,6843571.46,5891064.40,301.1,301.1,354.0,3309.65,765.54,33.239216,25.724837,28.994771,38.945098,30.313072,33.654902,12.119608,33.828758,201.326797,51.0,153
6,Ampara,2015,Maha,9130856.17,7369554.89,1266.4,1266.4,1696.0,3827.32,808.27,29.450235,24.316432,26.451174,34.610329,28.590141,30.868545,14.265728,32.959155,125.953052,63.0,213
7,Ampara,2015,Yala,6843510.92,5931859.69,401.2,401.2,478.0,3303.15,738.99,32.486928,25.521569,28.483007,38.645752,30.079085,33.431373,12.508497,32.341830,195.640523,51.0,153
8,Ampara,2016,Maha,9086788.72,7535898.34,825.7,825.7,1290.0,3999.87,854.76,30.255660,23.475472,26.400943,35.106132,27.342453,30.514623,13.152358,32.508019,165.070755,51.0,212
9,Ampara,2016,Yala,6843705.99,5682866.75,352.2,352.2,436.0,3214.45,738.57,33.143137,25.971895,29.031373,38.926797,30.645098,33.877778,12.936601,34.194118,195.300654,51.0,153


In [26]:
weather_output = PREPROCESSED_PATH / "02_weather_preprocessed.csv"
weather_season.to_csv(weather_output, index=False)

print("Saved:", weather_output)
print("Rows:", len(weather_season))

Saved: /Users/ravindudanthanarayana/Desktop/IT3091---Machine-Learning-Assignment/Preprocessed Datasets/02_weather_preprocessed.csv
Rows: 672


In [27]:
# Save this executed notebook (Colab only; locally just save with Ctrl/Cmd+S)
from getpass import getpass
import subprocess

MEMBER_NAME = 'Danthanarayana D.M.R'
NOTEBOOK_NAME = '02_IT24100533_Danthanarayana_DMR_Weather.ipynb'
FILES_TO_ADD = ['Preprocessed Datasets/02_weather_preprocessed.csv']

try:
    from google.colab import _message
except ImportError:
    _message = None
    print("Not running in Colab - skipping notebook export. Save the notebook from Jupyter/VS Code instead.")

if _message is not None:
    # Save the current Colab notebook inside the cloned repository
    current_notebook = _message.blocking_request("get_ipynb", request="", timeout_sec=10)
    notebook_path = REPO_PATH / NOTEBOOK_NAME

    with open(notebook_path, "w", encoding="utf-8") as f:
        json.dump(current_notebook["ipynb"], f, ensure_ascii=False, indent=1)

    print("Saved notebook:", notebook_path.name)

Not running in Colab - skipping notebook export. Save the notebook from Jupyter/VS Code instead.
